In [3]:
#!pip uninstall langchain -y && pip uninstall langchain-core -y && pip uninstall langchain-community -y
!pip install langchain==0.3.27 langchain-openai==0.3.33 langchain-community==0.3.24

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 12.0 MB/s  0:00:00
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
Using cached openai-1.109.1-py3-none-any.whl (948 kB)

  Attempting uninstall: openai

    Found existing installation: openai 2.38.0

   ---------------------------------------- 0/5 [openai]
   ---------------------------------------- 0/5 [openai]
   ---------------------------------------- 0/5 [openai]
   ---------------------------------------- 0/5 [openai]
    Uninstalling openai-2.38.0:
   ---------------------------------------- 0/5 [openai]
      Successfully uninstalled openai-2.38.0
   --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 0.41.1 requires langchain<=0.3,>0.2, but you have langchain 0.3.27 which is incompatible.
embedchain 0.1.116 requires langchain<=0.3,>0.2, but you have langchain 0.3.27 which is incompatible.
embedchain 0.1.116 requires langchain-community<0.3.0,>=0.2.6, but you have langchain-community 0.3.24 which is incompatible.
embedchain 0.1.116 requires langchain-openai<0.2.0,>=0.1.7, but you have langchain-openai 0.3.33 which is incompatible.
instructor 1.3.3 requires jiter<0.5.0,>=0.4.1, but you have jiter 0.14.0 which is incompatible.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-cohere 0.1.9

In [3]:
import os
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
import langchain, langchain_community

import utils
from langchain_core.prompts import ChatPromptTemplate


In [5]:
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


In [6]:
WEATHER_API_KEY = "6cb0bbb46d8a42b3b2371438262405"

In [7]:
@tool
def get_weather(city: str) -> str:
    """Fetch current weather using WeatherAPI."""
    if not WEATHER_API_KEY:
        return "Error: API key missing. Check your .env file."

    url = f"https://api.weatherapi.com/v1/current.json?key={WEATHER_API_KEY}&q={city}"
    response = requests.get(url)

    if response.status_code != 200:
        return f"API error: {response.text}"

    data = response.json()
    temp = data["current"]["temp_c"]
    cond = data["current"]["condition"]["text"]
    return f"Weather in {city}: {temp}°C, {cond}."

In [8]:
model = ChatOpenAI(model="gpt-4o-mini")

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert weather assistant. Use the tools provided to answer weather-related queries accurately."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [10]:
tools = [get_weather]
agent = create_tool_calling_agent(llm=model, tools=tools, prompt=prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=1)

In [11]:
query = "What's the weather in London right now?"
result = executor.invoke({"input": query})
print("Agent Output:", result)



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': 'London'}`


Weather in London: 18.1°C, Sunny.

> Finished chain.
Agent Output: {'input': "What's the weather in London right now?", 'output': 'Agent stopped due to max iterations.'}
